# Claude Advisor Tool — Workaround for Amazon Bedrock

## What is the Advisor Tool?

The [Claude Advisor Tool](https://docs.anthropic.com/en/docs/build-with-claude/advisor) is an Anthropic API feature that **pairs a strong model (Opus) as a planning advisor with a fast model (Sonnet) as the executor** — in a single API call.

The executor decides *when* to consult the advisor. When it does, the server automatically runs Opus with the full conversation transcript and returns its guidance. The executor continues generating, now informed.

```
┌─── Single /v1/messages request ───────────────────────┐
│                                                       │
│  Executor (Sonnet) ──▶ "I need guidance" ──▶ Server   │
│                                                │      │
│                                     Runs Advisor (Opus)│
│                                                │      │
│  Executor (Sonnet) ◀── advisor guidance ◀──────┘      │
│        │                                              │
│        ▼                                              │
│  Final response                                       │
└───────────────────────────────────────────────────────┘
```

### Where is it available?

| Platform | Available? |
|----------|------------|
| Anthropic API (direct) | ✅ Beta (`advisor-tool-2026-03-01`) |
| Claude Platform on AWS | ✅ |
| Amazon Bedrock (Converse / InvokeModel) | ❌ Not yet |

**The native server-side implementation is the better approach** — it's faster, simpler, and has built-in caching. This notebook implements a **feasible client-side workaround** for teams that need to stay on Amazon Bedrock today.

## Server-Side vs Client-Side: Why Native Wins

The native advisor tool handles everything server-side. It's the superior approach — this workaround is a bridge until Bedrock supports it natively.

| | Server-Side (Native) | Client-Side (This Workaround) |
|---|---|---|
| **Latency** | Single request — advisor runs in-process, no network round-trip between models | Extra round-trip per advisor call (client → Bedrock → client → Bedrock) |
| **Simplicity** | One API call, one response | Agentic loop managing multiple calls + retry logic |
| **Transcript handling** | Server automatically constructs the advisor's view from the full context | Client must serialize and pass the full conversation manually (token overhead) |
| **Caching** | Built-in ephemeral caching with configurable TTL | Must implement separately via Bedrock prompt caching (see below) |
| **Token visibility** | `usage.iterations[]` gives clean per-role breakdown | Must track manually across calls |
| **Atomicity** | Single request = no partial failure between executor and advisor | Advisor call can fail mid-loop, requiring retry/recovery logic |
| **Error handling** | Server manages retries internally | Client must handle timeouts, throttling, and partial failures |

### The Caching Consideration (Key Difference)

This is the most impactful difference between server-side and client-side.

**Native advisor tool caching:**
- The advisor declaration accepts a `caching` parameter: `{ "type": "ephemeral", "ttl": "5m" }`
- The server caches the advisor model's KV state (the processed transcript) automatically
- When the executor consults the advisor multiple times within a session, subsequent calls reuse the cached context
- You pay the cache *write* premium (25% over base) once, then get 90% off input tokens on subsequent hits
- For a typical advisor call with a 5K-token transcript, the second+ calls cost ~$0.0075 instead of ~$0.075 (Opus rates)
- The TTL refreshes on each hit — as long as the advisor is called within 5 minutes, the cache stays warm

**Client-side caching (this workaround):**
- No automatic caching between the executor and advisor calls
- Each advisor call sends the full transcript as new input → full input token cost every time
- **Mitigation:** You can use [Bedrock prompt caching](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html) by adding `cachePoint` markers to the advisor's system prompt or conversation prefix
- Bedrock prompt caching requires a minimum prefix of **2,048 tokens for Opus** (1,024 for Sonnet)
- Bedrock TTL options: 5 minutes (default) or up to 1 hour (extended, with 2× write premium)
- The cache key is based on byte-identical prefix matching — any change in the transcript before the cache point invalidates it

**Practical impact:** In a multi-turn session where the advisor is consulted 3 times:

| Scenario | Advisor input cost (Opus @ $15/M) |
|----------|-----------------------------------|
| Native with caching (5K transcript) | Write: $0.094 + 2 reads: $0.015 = **$0.109** |
| Client-side, no caching | 3 × $0.075 = **$0.225** |
| Client-side with Bedrock prompt caching | Write: $0.094 + 2 reads: $0.015 = **~$0.109** |

With Bedrock prompt caching, you can close the cost gap — but it requires careful implementation. The system prompt and early conversation turns must remain byte-identical across advisor calls, and the growing transcript (which changes each turn) will fall outside the cached prefix.

### So why use client-side?

**Because Bedrock doesn't support the native tool yet.** If you need to stay on Bedrock (for VPC networking, AWS IAM, compliance, consolidated billing, or existing infrastructure), this workaround gives you the same *pattern* and similar *cost profile* — you just accept the extra latency, complexity, and manual caching setup.

When Bedrock adds native advisor tool support, migration is straightforward:
1. Remove the orchestration loop
2. Add the advisor to your tool list with `type: advisor_20260301`
3. Switch to a single `converse()` call
4. Remove manual caching logic (native handles it)

The architecture stays the same — only the plumbing changes.

## Setup

In [ ]:
%pip install boto3 --quiet

In [ ]:
import json
import boto3

# Configuration — adjust to your region and model access
REGION = "us-east-1"
EXECUTOR_MODEL = "us.anthropic.claude-sonnet-4-6-v1"  # Fast, cheap
ADVISOR_MODEL = "us.anthropic.claude-opus-4-7-v1"     # Strong, expensive

client = boto3.client("bedrock-runtime", region_name=REGION)
print(f"✓ Bedrock client ready ({REGION})")
print(f"  Executor: {EXECUTOR_MODEL}")
print(f"  Advisor:  {ADVISOR_MODEL}")

## Step 1: Define the Advisor Tool

We give the executor a tool called `consult_advisor`. The tool description tells the model **when** to use it — following Anthropic's own guidance:

1. **Before substantive work** (not orientation — actual writing/building)
2. **When stuck** (recurring errors, approach not converging)
3. **Before declaring done** (final review)
4. **On disagreement** (when evidence contradicts prior advisor guidance)

In [ ]:
ADVISOR_TOOL = {
    "toolSpec": {
        "name": "consult_advisor",
        "description": (
            "Consult a senior advisor for strategic guidance. The advisor sees "
            "the full conversation and provides expert-level review. "
            "Call this: (1) before substantive work begins, "
            "(2) when stuck or an approach isn't converging, "
            "(3) before declaring done for a final review, "
            "(4) when your evidence contradicts prior advice — surface the conflict."
        ),
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "What specific guidance do you need?"
                    }
                },
                "required": ["question"]
            }
        }
    }
}

print("✓ Advisor tool defined")
print(json.dumps(ADVISOR_TOOL, indent=2))

## Step 2: Define System Prompts

Two models, two roles, two prompts.

In [ ]:
EXECUTOR_SYSTEM = (
    "You are a capable AI assistant. You have access to a senior advisor "
    "via the consult_advisor tool. Use it for complex decisions — don't "
    "consult for trivial questions, but do consult before major design "
    "decisions or when you're unsure about the best approach."
)

ADVISOR_SYSTEM = (
    "You are a senior technical advisor. Review the executor's work and "
    "provide concise, actionable guidance. Focus on: correctness, approach "
    "quality, edge cases missed, and strategic direction. "
    "Keep responses under 500 words. Be direct."
)

## Step 3: The Orchestration Loop

This is the core workaround. The loop:
1. Calls the executor (Sonnet) with the conversation + advisor tool
2. If the executor calls `consult_advisor`, intercepts and routes to the advisor (Opus)
3. Returns the advisor's guidance as a tool result
4. Repeats until the executor produces a final response

**Note on caching:** To reduce cost on repeated advisor calls, we place the system prompt as a stable prefix. In production, you'd add a `cachePoint` marker after the system block to enable Bedrock prompt caching (requires 2,048+ token prefix for Opus).

In [ ]:
def run_advisor_pattern(user_message: str, max_advisor_calls: int = 3) -> dict:
    """
    Run the advisor-executor pattern on Bedrock.
    
    Returns dict with 'response', 'usage', and 'advisor_calls'.
    """
    messages = [{"role": "user", "content": [{"text": user_message}]}]
    usage = {"executor": {"in": 0, "out": 0}, "advisor": {"in": 0, "out": 0}}
    advisor_calls = 0

    while True:
        # --- Call Executor (Sonnet) ---
        response = client.converse(
            modelId=EXECUTOR_MODEL,
            messages=messages,
            system=[{"text": EXECUTOR_SYSTEM}],
            toolConfig={"tools": [ADVISOR_TOOL]},
            inferenceConfig={"maxTokens": 4096},
        )

        # Track executor usage
        u = response.get("usage", {})
        usage["executor"]["in"] += u.get("inputTokens", 0)
        usage["executor"]["out"] += u.get("outputTokens", 0)

        # Add assistant message to history
        assistant_msg = response["output"]["message"]
        messages.append(assistant_msg)

        # If executor finished naturally, we're done
        if response["stopReason"] == "end_turn":
            break

        # If executor wants to use a tool
        if response["stopReason"] == "tool_use":
            tool_results = []

            for block in assistant_msg.get("content", []):
                if "toolUse" not in block:
                    continue

                tool = block["toolUse"]
                if tool["name"] != "consult_advisor":
                    continue

                # --- Call Advisor (Opus) ---
                if advisor_calls >= max_advisor_calls:
                    advice = "[Advisor limit reached. Proceed with your best judgment.]"
                else:
                    question = tool["input"].get("question", "")
                    print(f"  🧠 Advisor consulted: {question[:80]}...")

                    # The advisor sees the full transcript + the specific question.
                    # In the native tool, the server constructs this view automatically.
                    # Here we must do it manually.
                    advisor_response = client.converse(
                        modelId=ADVISOR_MODEL,
                        messages=[{
                            "role": "user",
                            "content": [{
                                "text": (
                                    f"## Full conversation so far\n\n"
                                    f"{json.dumps(messages, indent=2, default=str)}\n\n"
                                    f"---\n\n"
                                    f"## Executor's question\n{question}"
                                )
                            }]
                        }],
                        system=[{"text": ADVISOR_SYSTEM}],
                        inferenceConfig={"maxTokens": 1024},
                    )

                    # Track advisor usage
                    au = advisor_response.get("usage", {})
                    usage["advisor"]["in"] += au.get("inputTokens", 0)
                    usage["advisor"]["out"] += au.get("outputTokens", 0)
                    advisor_calls += 1

                    # Extract advisor text
                    advice = ""
                    for b in advisor_response["output"]["message"]["content"]:
                        if "text" in b:
                            advice += b["text"]

                    print(f"  ✓ Advisor responded ({len(advice)} chars)")

                tool_results.append({
                    "toolResult": {
                        "toolUseId": tool["toolUseId"],
                        "content": [{"text": advice}]
                    }
                })

            # Return advisor guidance to executor
            messages.append({"role": "user", "content": tool_results})
        else:
            break  # Unknown stop reason

    # Extract final text
    final_text = ""
    for block in assistant_msg.get("content", []):
        if "text" in block:
            final_text += block["text"]

    return {
        "response": final_text,
        "usage": usage,
        "advisor_calls": advisor_calls,
    }

print("✓ Orchestration function defined")

## Step 4: Run It

Let's give the executor a complex task where consulting an advisor adds real value.

In [ ]:
task = """
Design a serverless event-driven architecture for a real-time fraud 
detection system processing 10,000 transactions per second.

Requirements:
- Sub-100ms latency for scoring
- ML model inference (XGBoost + neural network ensemble)
- Real-time feature engineering from streaming data
- Explainability for flagged transactions
- 99.99% availability

Provide the architecture with AWS services, data flow, and key decisions.
"""

print("🚀 Running advisor-executor pattern...\n")
result = run_advisor_pattern(task)

print(f"\n{'─' * 60}")
print(result["response"])

## Step 5: Cost Breakdown

The whole point of the advisor pattern is **cost efficiency**: Opus only for the ~400-700 token planning moments, Sonnet for the bulk generation.

In [ ]:
print("─── Token Usage ────────────────────────────────────────")
print(f"  Executor (Sonnet):  in={result['usage']['executor']['in']:,}  out={result['usage']['executor']['out']:,}")
print(f"  Advisor  (Opus):    in={result['usage']['advisor']['in']:,}  out={result['usage']['advisor']['out']:,}")
print(f"  Advisor calls:      {result['advisor_calls']}")
print("────────────────────────────────────────────────────────")

# Rough cost estimate (Bedrock on-demand pricing)
SONNET_IN = 3.00 / 1_000_000   # $3/M input tokens
SONNET_OUT = 15.00 / 1_000_000  # $15/M output tokens
OPUS_IN = 15.00 / 1_000_000    # $15/M input tokens
OPUS_OUT = 75.00 / 1_000_000   # $75/M output tokens

executor_cost = (
    result['usage']['executor']['in'] * SONNET_IN +
    result['usage']['executor']['out'] * SONNET_OUT
)
advisor_cost = (
    result['usage']['advisor']['in'] * OPUS_IN +
    result['usage']['advisor']['out'] * OPUS_OUT
)

total = executor_cost + advisor_cost
all_opus = (
    (result['usage']['executor']['in'] + result['usage']['advisor']['in']) * OPUS_IN +
    (result['usage']['executor']['out'] + result['usage']['advisor']['out']) * OPUS_OUT
)

print(f"\n  💰 Estimated cost:")
print(f"     Executor (Sonnet): ${executor_cost:.4f}")
print(f"     Advisor  (Opus):   ${advisor_cost:.4f}")
print(f"     Total:             ${total:.4f}")
print(f"\n  📊 vs. all-Opus:      ${all_opus:.4f} ({total/all_opus*100:.0f}% of all-Opus cost)")
print(f"     Savings:           ${all_opus - total:.4f} ({(1 - total/all_opus)*100:.0f}% reduction)")

## Step 6 (Optional): Adding Bedrock Prompt Caching

To close the cost gap with the native advisor tool's built-in caching, you can enable Bedrock prompt caching on the advisor calls.

### How native caching works (server-side)

In the native advisor tool, you declare caching in the tool definition:
```json
{
  "type": "advisor_20260301",
  "name": "advisor",
  "model": "claude-opus-4-7",
  "caching": { "type": "ephemeral", "ttl": "5m" }
}
```

The server caches the advisor's processed KV state (the full transcript prefix). On subsequent advisor calls within the same request, the cached prefix is reused — you only pay 10% of input token cost for the cached portion.

### Replicating with Bedrock prompt caching (client-side)

On Bedrock, prompt caching works via `cachePoint` markers in the message structure. The idea:
- Place the **advisor system prompt** (stable across calls) as a cacheable prefix
- The growing conversation transcript (changes each call) falls outside the cache

**Constraints:**
- Minimum cacheable prefix: **2,048 tokens** for Opus, 1,024 for Sonnet
- Prefix must be **byte-identical** — any change invalidates the cache
- Default TTL: 5 minutes (refreshes on hit). Extended: up to 1 hour (2× write premium)
- Cache writes cost 25% more than base input; reads cost 90% less

**The challenge:** In our workaround, the advisor's input is mostly the conversation transcript, which *grows* with each call. The system prompt alone may not hit the 2,048 token minimum. To work around this:

1. **Pad the advisor system prompt** with detailed instructions/examples to exceed 2,048 tokens
2. **Or** structure the advisor input so the first message contains a stable reference document (e.g., project requirements) that exceeds the threshold
3. The growing transcript goes in subsequent messages (outside the cached prefix)

This means you can cache the system prompt + stable context, but the growing transcript will always be billed at full input rates. The native tool doesn't have this limitation because it caches the full processed state server-side.

In [ ]:
# Example: Advisor call with Bedrock prompt caching enabled
# (Illustrative — cachePoint support depends on your Bedrock API version)

def call_advisor_with_caching(messages, question, stable_context=""):
    """
    Call the advisor with prompt caching enabled.
    
    The stable_context (e.g., project requirements, architecture docs)
    goes before the cachePoint. The growing transcript goes after.
    """
    # Build system with cache point after stable content
    system_blocks = [
        {"text": ADVISOR_SYSTEM},
    ]
    
    # If we have stable context that exceeds 2048 tokens, cache it
    if stable_context:
        system_blocks.append({"text": stable_context})
    
    # Add cache point after stable prefix
    system_blocks.append({"cachePoint": {"type": "default"}})
    
    advisor_response = client.converse(
        modelId=ADVISOR_MODEL,
        messages=[{
            "role": "user",
            "content": [{
                "text": (
                    f"## Conversation transcript\n\n"
                    f"{json.dumps(messages, indent=2, default=str)}\n\n"
                    f"---\n\n"
                    f"## Question\n{question}"
                )
            }]
        }],
        system=system_blocks,
        inferenceConfig={"maxTokens": 1024},
    )
    
    # Check cache hit in response usage
    u = advisor_response.get("usage", {})
    cache_read = u.get("cacheReadInputTokens", 0)
    cache_write = u.get("cacheWriteInputTokens", 0)
    
    if cache_read > 0:
        print(f"  💾 Cache HIT: {cache_read} tokens read from cache (90% discount)")
    elif cache_write > 0:
        print(f"  📝 Cache WRITE: {cache_write} tokens cached (25% premium, will save on next call)")
    
    return advisor_response

print("✓ Caching-enabled advisor function defined")
print("  Note: Effective when stable_context exceeds 2,048 tokens")

## Summary

### Recommendation: Use the native advisor tool when possible

The server-side implementation wins on every dimension:
- **Faster** — no extra network round-trips
- **Simpler** — one API call, no orchestration code
- **Cheaper** — built-in caching with automatic KV state reuse
- **More reliable** — atomic execution, no partial failures

### This workaround is feasible when you must stay on Bedrock

Valid reasons to use client-side:
- VPC networking requirements (PrivateLink to Bedrock)
- AWS IAM-only auth (no Anthropic API keys)
- Compliance (data stays within AWS boundary)
- Consolidated billing through AWS
- Existing Bedrock infrastructure/quotas

### Migration path

When Bedrock adds native advisor support:
1. Delete the orchestration loop
2. Add `advisor_20260301` to your tool list
3. Single `converse()` call
4. Remove manual caching → native handles it

The *thinking* stays the same — strong model for planning, fast model for execution. Only the plumbing changes.